In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

In [3]:
df = pd.read_csv("/home/vlad/eQTL/processed/classification_with_orf.tsv", sep="\t")

print(df.shape)
print(df["structural_category"].value_counts())

(489545, 57)
structural_category
full-splice_match          204341
novel_not_in_catalog        99242
novel_in_catalog            85493
incomplete-splice_match     67457
genic_intron                12043
antisense                    6639
genic                        5371
intergenic                   4844
fusion                       4115
Name: count, dtype: int64


In [4]:
TARGET = "structural_category"

NUMERIC_FEATURES = [
    "length",
    "exons",
    "perc_A_downstream_TTS",
    "protein_length",
    "has_orf",
    "diff_to_gene_TSS",
    "diff_to_gene_TTS",
    "ref_exons",
    "diff_to_TSS",
    "diff_to_TTS",
    "ref_length",
]

CATEGORICAL_FEATURES = [
    "strand",
    "FSM_class",
    "RTS_stage",
    "subcategory",
    "orf_type",
    "all_canonical",
    "bite",
]

for col in NUMERIC_FEATURES:
    nan_pct = df[col].isna().mean() * 100
    print(f"  {col}: {nan_pct:.1f}% NaN")

for col in CATEGORICAL_FEATURES:
    nan_pct = df[col].isna().mean() * 100
    n_unique = df[col].nunique()
    print(f"  {col}: {nan_pct:.1f}% NaN, {n_unique} unique")

  length: 0.0% NaN
  exons: 0.0% NaN
  perc_A_downstream_TTS: 0.0% NaN
  protein_length: 0.0% NaN
  has_orf: 0.0% NaN
  diff_to_gene_TSS: 3.4% NaN
  diff_to_gene_TTS: 3.4% NaN
  ref_exons: 5.4% NaN
  diff_to_TSS: 44.5% NaN
  diff_to_TTS: 44.5% NaN
  ref_length: 44.5% NaN
  strand: 0.0% NaN, 2 unique
  FSM_class: 0.0% NaN, 3 unique
  RTS_stage: 0.0% NaN, 2 unique
  subcategory: 0.0% NaN, 14 unique
  orf_type: 0.0% NaN, 5 unique
  all_canonical: 12.0% NaN, 2 unique
  bite: 12.0% NaN, 2 unique


In [5]:
df_proc = df.copy()

for col in ["diff_to_TSS", "diff_to_TTS", "ref_length", 
            "diff_to_gene_TSS", "diff_to_gene_TTS", "ref_exons"]:
    median_val = df_proc[col].median()
    df_proc[col] = df_proc[col].fillna(median_val)
    print(f"{col}: with median {median_val:.1f}")

df_proc["all_canonical"] = df_proc["all_canonical"].fillna("mono_exon")
df_proc["bite"] = df_proc["bite"].fillna("mono_exon")

print(df_proc["all_canonical"].value_counts())
print(df_proc["bite"].value_counts())

diff_to_TSS: with median 0.0
diff_to_TTS: with median 0.0
ref_length: with median 1356.0
diff_to_gene_TSS: with median 0.0
diff_to_gene_TTS: with median 0.0
ref_exons: with median 9.0
all_canonical
canonical        423285
mono_exon         58969
non_canonical      7291
Name: count, dtype: int64
bite
False        367781
True          62795
mono_exon     58969
Name: count, dtype: int64


In [6]:
for col in df_proc.columns:
    nan_pct = df[col].isna().mean() * 100  
    dtype = df_proc[col].dtype
    n_unique = df_proc[col].nunique()
    print(f"  {col:35s} dtype={str(dtype):10s} NaN={nan_pct:.0f}% unique={n_unique}")

  isoform                             dtype=object     NaN=0% unique=489545
  chrom                               dtype=object     NaN=0% unique=50
  start                               dtype=int64      NaN=0% unique=253475
  end                                 dtype=int64      NaN=0% unique=253074
  strand                              dtype=object     NaN=0% unique=2
  length                              dtype=int64      NaN=0% unique=11899
  exons                               dtype=int64      NaN=0% unique=121
  structural_category                 dtype=object     NaN=0% unique=9
  subcategory                         dtype=object     NaN=0% unique=14
  FSM_class                           dtype=object     NaN=0% unique=3
  associated_gene                     dtype=object     NaN=0% unique=83365
  associated_transcript               dtype=object     NaN=0% unique=207575
  ref_length                          dtype=float64    NaN=44% unique=9861
  ref_exons                           dty

In [7]:
DROP_COLS = [
    'min_sample_cov', 'min_cov', 'min_cov_pos', 'sd_cov', 'FL',
    'n_indels', 'n_indels_junc', 'iso_exp', 'gene_exp', 'ratio_exp',
    'ORF_length', 'CDS_length', 'CDS_start', 'CDS_end',
    'CDS_genomic_start', 'CDS_genomic_end', 'psauron_score',
    'CDS_type', 'predicted_NMD', 'dist_to_CAGE_peak', 'within_CAGE_peak',
    'dist_to_polyA_site', 'within_polyA_site', 'polyA_motif',
    'polyA_dist', 'polyA_motif_found', 'ratio_TSS', 'ref_num_exons',
    
    'coding',
    
    'isoform', 'associated_gene', 'associated_transcript',
    
    'start', 'end', 'chrom',
    
    'seq_A_downstream_TTS',
    
    'diff_to_TSS', 'diff_to_TTS',
    'diff_to_TSS_genomic', 'diff_to_TTS_genomic',
]

df_clean = df_proc.drop(columns=DROP_COLS)

print(f"Before: {df_proc.shape[1]}")
print(f"After: {df_clean.shape[1]}")
print(f"\nColumns:")
for col in df_clean.columns:
    print(f"  {col}")

Before: 57
After: 17

Columns:
  strand
  length
  exons
  structural_category
  subcategory
  FSM_class
  ref_length
  ref_exons
  diff_to_gene_TSS
  diff_to_gene_TTS
  RTS_stage
  all_canonical
  bite
  perc_A_downstream_TTS
  orf_type
  protein_length
  has_orf


In [8]:
df_clean.head(100)

,strand,length,exons,structural_category,subcategory,FSM_class,ref_length,ref_exons,diff_to_gene_TSS,diff_to_gene_TTS,RTS_stage,all_canonical,bite,perc_A_downstream_TTS,orf_type,protein_length,has_orf
0,-,1773,10,novel_not_in_catalog,at_least_one_novel_splicesite,C,1356.0,11.0,0.0,0.0,False,canonical,False,30.0,complete,99.0,1
1,-,2643,8,novel_not_in_catalog,intron_retention,C,1356.0,11.0,0.0,0.0,False,canonical,True,30.0,complete,121.0,1
2,-,2290,11,novel_not_in_catalog,at_least_one_novel_splicesite,C,1356.0,11.0,0.0,0.0,True,canonical,False,30.0,complete,99.0,1
3,-,4150,8,novel_not_in_catalog,at_least_one_novel_splicesite,C,1356.0,11.0,-8441.0,0.0,False,canonical,False,30.0,complete,128.0,1
4,-,2824,9,novel_not_in_catalog,intron_retention,C,1356.0,11.0,0.0,0.0,True,canonical,False,30.0,complete,99.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,+,6909,1,antisense,mono-exon,A,1356.0,9.0,-103.0,459.0,False,mono_exon,mono_exon,75.0,no_orf,0.0,0
96,+,3136,1,intergenic,mono-exon,A,1356.0,9.0,0.0,0.0,False,mono_exon,mono_exon,60.0,no_orf,0.0,0
97,-,4380,1,genic,mono-exon,C,1356.0,9.0,-3507.0,-3491.0,False,mono_exon,mono_exon,10.0,complete,152.0,1
98,+,2594,3,antisense,multi-exon,A,1356.0,3.0,-3491.0,8202.0,False,canonical,True,25.0,complete,197.0,1


In [9]:
print(pd.crosstab(df['structural_category'], df['subcategory']))

subcategory              3prime_fragment  5prime_fragment  alternative_3end  \
structural_category                                                           
antisense                              0                0                 0   
full-splice_match                      0                0              4546   
fusion                                 0                0                 0   
genic                                  0                0                 0   
genic_intron                           0                0                 0   
incomplete-splice_match            47643             6419                 0   
intergenic                             0                0                 0   
novel_in_catalog                       0                0                 0   
novel_not_in_catalog                   0                0                 0   

subcategory              alternative_3end5end  alternative_5end  \
structural_category                                            

In [11]:
mapping = {
    ('full-splice_match', 'reference_match'):              'real_high',
    ('full-splice_match', 'alternative_3end'):             'real_high',
    ('full-splice_match', 'alternative_5end'):             'real_high',
    ('full-splice_match', 'alternative_3end5end'):         'real_high',
    ('novel_in_catalog',  'combination_of_known_junctions'): 'real_high',
    ('novel_in_catalog',  'combination_of_known_splicesites'): 'real_mid',

    ('full-splice_match', 'mono-exon'):                    'real_mid',
    ('novel_in_catalog',  'intron_retention'):             'uncertain',
    ('novel_in_catalog',  'mono-exon_by_intron_retention'): 'uncertain',
    ('novel_not_in_catalog', 'at_least_one_novel_splicesite'): 'uncertain',
    ('incomplete-splice_match', '3prime_fragment'):        'uncertain',
    ('incomplete-splice_match', '5prime_fragment'):        'uncertain',
    ('incomplete-splice_match', 'internal_fragment'):      'uncertain',

    ('novel_not_in_catalog', 'intron_retention'):          'artifact',
    ('incomplete-splice_match', 'intron_retention'):       'artifact',
    ('incomplete-splice_match', 'mono-exon'):              'artifact',
    ('fusion', 'intron_retention'):                        'artifact',
    ('fusion', 'multi-exon'):                              'artifact',
    ('genic_intron', 'mono-exon'):                         'artifact',
    ('genic_intron', 'multi-exon'):                        'artifact',
    ('genic', 'mono-exon'):                                'artifact',
    ('genic', 'multi-exon'):                               'artifact',
    ('intergenic', 'mono-exon'):                           'artifact',
    ('intergenic', 'multi-exon'):                          'artifact',
    ('antisense', 'mono-exon'):                            'artifact',
    ('antisense', 'multi-exon'):                           'artifact',
}

In [12]:
df['target'] = list(zip(df['structural_category'], df['subcategory']))
df['target'] = df['target'].map(mapping)

print(df['target'].isna().sum())  
print(df['target'].value_counts())

DROP_FROM_X = ['structural_category', 'subcategory']

0
target
real_high    202421
uncertain    175559
artifact      66905
real_mid      44660
Name: count, dtype: int64


In [13]:
df['target_4class'] = df['target'].copy()

df['target_3class'] = df['target'].replace({
    'real_mid': 'real',
    'real_high': 'real'
})

print(df['target_4class'].value_counts())
print(df['target_3class'].value_counts())

target_4class
real_high    202421
uncertain    175559
artifact      66905
real_mid      44660
Name: count, dtype: int64
target_3class
real         247081
uncertain    175559
artifact      66905
Name: count, dtype: int64


In [14]:
df_model = df_clean.copy()

df_model = df_model.drop(columns=['structural_category', 'subcategory'])

df_model['target_4class'] = df['target_4class']
df_model['target_3class'] = df['target_3class']

df_model['strand'] = df_model['strand'].map({'+': 1, '-': 0})
df_model['RTS_stage'] = df_model['RTS_stage'].astype(int)
df_model['all_canonical'] = df_model['all_canonical'].map({
    'canonical': 2,
    'non_canonical': 1,
    'mono_exon': 0
})
df_model['bite'] = df_model['bite'].map({
    'True': 1,
    'False': 0,
    'mono_exon': -1
})

df_model = pd.get_dummies(df_model,
                        columns=['FSM_class', 'orf_type'],
                        dtype=int)

from sklearn.preprocessing import StandardScaler

numeric_cols = ['length', 'exons', 'perc_A_downstream_TTS',
                'protein_length', 'diff_to_gene_TSS',
                'diff_to_gene_TTS', 'ref_exons',
                'ref_length']

scaler = StandardScaler()
df_model[numeric_cols] = scaler.fit_transform(df_model[numeric_cols])

print(df_model.shape)
print(df_model.columns.tolist())

(489545, 23)
['strand', 'length', 'exons', 'ref_length', 'ref_exons', 'diff_to_gene_TSS', 'diff_to_gene_TTS', 'RTS_stage', 'all_canonical', 'bite', 'perc_A_downstream_TTS', 'protein_length', 'has_orf', 'target_4class', 'target_3class', 'FSM_class_A', 'FSM_class_B', 'FSM_class_C', 'orf_type_3prime_partial', 'orf_type_5prime_partial', 'orf_type_complete', 'orf_type_internal', 'orf_type_no_orf']


In [ ]:
out_path = "/home/vlad/eQTL/processed/df_model.csv"
df_model.to_csv(out_path, index=False)